# Demo 04 : Command and Send

Run the cells in order and watch. There is nothing to write.

## a · command

We return a state update and the next node from the same node.

In [1]:
from typing import Literal, TypedDict
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command

class Visit(TypedDict):
    priority: str
    assigned: str

def triage(state: Visit) -> Command[Literal["urgent", "routine"]]:
    target = "urgent" if state["priority"] == "urgent" else "routine"
    return Command(update={"assigned": target}, goto=target)

builder = StateGraph(Visit)
builder.add_node("triage", triage)
builder.add_node("urgent", lambda s: {})
builder.add_node("routine", lambda s: {})
builder.add_edge(START, "triage")
builder.add_edge("urgent", END)
builder.add_edge("routine", END)
print("route:", builder.compile().invoke({"priority": "urgent", "assigned": ""})["assigned"])

route: urgent


## b · send-map-reduce

We create one runtime branch per depot and collect each deterministic report.

In [2]:
import operator
from typing import Annotated, TypedDict
from langgraph.graph import END, START, StateGraph
from langgraph.types import Send

class Summary(TypedDict):
    depots: list[str]
    reports: Annotated[list[str], operator.add]
    summary: str

class Depot(TypedDict):
    name: str

def fan_out(state: Summary):
    return [Send("report", {"name": name}) for name in state["depots"]]

def report(state: Depot) -> dict:
    return {"reports": [f"{state['name']}: checked"]}

def collect(state: Summary) -> dict:
    return {"summary": f"{len(state['reports'])} depots checked"}

builder = StateGraph(Summary)
builder.add_node("report", report)
builder.add_node("collect", collect)
builder.add_conditional_edges(START, fan_out, ["report"])
builder.add_edge("report", "collect")
builder.add_edge("collect", END)
result = builder.compile().invoke({"depots": ["Rietpoort", "Broekdam", "Veldhaven"], "reports": [], "summary": ""})
print("reports:", sorted(result["reports"]))
print("summary:", result["summary"])

reports: ['Broekdam: checked', 'Rietpoort: checked', 'Veldhaven: checked']
summary: 3 depots checked


## c · deferred-nodes

Two branches of different length join at `collect` : `short` is one step,
`long1` then `long2` are two. With plain edges `collect` runs as soon as any
branch reaches it. With `defer=True` it waits until no other work is left.

In [3]:
class Log(TypedDict):
    log: Annotated[list[str], operator.add]

def uneven(defer: bool):
    g = StateGraph(Log)
    g.add_node("short", lambda s: {"log": ["short"]})
    g.add_node("long1", lambda s: {"log": ["long1"]})
    g.add_node("long2", lambda s: {"log": ["long2"]})
    g.add_node("collect", lambda s: {"log": [f"collect saw {len(s['log'])}"]}, defer=defer)
    g.add_edge(START, "short")
    g.add_edge(START, "long1")
    g.add_edge("long1", "long2")
    g.add_edge("short", "collect")
    g.add_edge("long2", "collect")
    g.add_edge("collect", END)
    return g.compile()

print("plain   :", uneven(False).invoke({"log": []})["log"])
print("deferred:", uneven(True).invoke({"log": []})["log"])

plain   : ['long1', 'short', 'collect saw 2', 'long2', 'collect saw 4']
deferred: ['long1', 'short', 'long2', 'collect saw 3']


The plain `collect` runs twice and the first run sees only two entries. The
deferred `collect` runs once, after both branches. The same flag works on a
`Send` fan-out :

In [4]:
deferred = StateGraph(Summary)
deferred.add_node("report", report)
deferred.add_node("collect", collect, defer=True)
deferred.add_conditional_edges(START, fan_out, ["report"])
deferred.add_edge("report", "collect")
deferred.add_edge("collect", END)
result = deferred.compile().invoke({"depots": ["Rietpoort", "Broekdam"], "reports": [], "summary": ""})
print("deferred summary:", result["summary"])

deferred summary: 2 depots checked


## Things to try

- Add another depot to the map-reduce input
- Change the collected report format and run both graphs again